In [1]:
import json
import os
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool

In [2]:
from dotenv import load_dotenv
_ = load_dotenv()

import os 
API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
from crewai.llms.providers.openai.completion import OpenAICompletion
llm = OpenAICompletion(model="gpt-4o", api_key=API_KEY)


In [4]:
csv_carteira = CSVSearchTool(csv="ativos.csv")

Agente 1 - Gerente do cliente

In [5]:
gerente_cliente = Agent(
    role="Gerente de Carteira do Cliente",
    goal="Obtenha a pergunta do cliente sobre o ativo {ticket} e pesquise as ações no arquivo CSV da carteira do cliente",
    backstory="""
    Você é o gerente de clientes da carteira de investimentos do cliente.
    Você é o primeiro contato do cliente e fornece as informações para as demais análises com o ticket do ativo e informações de carteira necessárias
    """,
    verbose=True,
    max_iter=5,
    tools=[csv_carteira],
    memory=True
)

In [6]:
obter_carteira_cliente = Task(
    description=""",
    Use a pergunta do cliente e encontre o ativo {ticket} no arquivo CSV.
    Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o número total de ações em posse.
    """,
    expected_output="Se o cliente possuir os ativos, forneça o preço médio e o total de ações dos ativos",
    agent=gerente_cliente
)

In [7]:
import yfinance as yf

def pega_preco_ativo(ticket):
    data_final = datetime.today()
    data_inicial = data_final - timedelta(days=365)
    ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
    return ativo

In [8]:
resultado = pega_preco_ativo("PETR4.SA")
resultado

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA
Date,,,,,
2025-05-07,27.505302,27.514382,27.151156,27.514382,35050400
2025-05-08,27.886686,28.177269,27.650591,27.777719,44665000
2025-05-09,28.068302,28.313480,27.823124,28.240835,25075600
2025-05-12,28.740273,29.212465,28.740273,28.967289,53493600
2025-05-13,29.176142,29.285109,28.467850,28.849238,56165300
...,...,...,...,...,...
2026-04-29,48.959999,49.299999,48.000000,48.099998,47685000
2026-04-30,49.080002,49.380001,48.290001,48.810001,36666100


Agente 2 - Analista de Ações

In [9]:
analista_acoes = Agent(
    role="Analista senior de preço de ações",
    goal="Encontre o preço da ação {ticket} e analise suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago",
    backstory="""
    Você é um analista de  muito experiente.
    Você é responsável por analisar os ativos e fornecer informações sobre o preço do ativo para o gerente de  e fazer previsões sobre seu preço futuro.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [10]:
from crewai.tools import BaseTool
from pydantic import Field

In [11]:
class YahooFinanceTool(BaseTool):
    name: str = "Yahoo Finance Tool"
    description: str = "Use esta ferramenta para obter informações sobre o preço de um ativo, no último ano, usando a biblioteca yfinance. Forneça o ticket do ativo para obter as informações necessárias."

    def _run(self, ticket: str):
        """Executa a busca de preços de ações para o ativo fornecido usando a biblioteca yfinance."""
        try:
            data_final = datetime.today()
            data_inicial = data_final - timedelta(days=365)
            ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
            return ativo.to_dict()
        except Exception as e:
            return f"Erro ao obter dados do Yahoo Finance: {str(e)}"

In [12]:
yfinance_tool = YahooFinanceTool()
type(yfinance_tool)

__main__.YahooFinanceTool

In [13]:
obter_preco_acao = Task(
    description="""
    Use a ferramenta Yahoo Finance Tool para obter o preço da ação {ticket} e analisar suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago.
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago e especifique uma tendencia atual do preço da acao tanto para cima quanto para baixo",
    agent=analista_acoes,
    tools=[yfinance_tool]
)

Agente 3 - Analista de notícias

In [14]:
analista_noticias = Agent(
    role="Analista senior de notícias de ações",
    goal="Encontre as últimas notícias sobre o ativo {ticket} e analise seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira",
    backstory="""
    Você é um analista muito experiente.
    Você é responsável por analisar as notícias relacionadas aos ativos e fornecer informações sobre o impacto potencial dessas notícias no preço do ativo para o gerente de carteira.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [15]:
from langchain_community.tools import DuckDuckGoSearchResults
searchTool = DuckDuckGoSearchResults(backend="news", num_results=10)

In [16]:
obter_noticias = Task(
    description=f"""
    Use a ferramenta DuckDuckGo News Tool para obter as últimas notícias sobre o ativo e analisar seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira.
    A data atual é {datetime.now()}
    Componha os resultados em um relatório útil
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de analisar o impacto potencial das notícias no preço do ativo.",
    agent=analista_noticias,
    tool=[searchTool]
)

Agente 4 - Analista chefe de ações

In [17]:
analista_chefe = Agent(
    role="Analista chefe de investimentos",
    goal="Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.",
    backstory="""
    Você é um analista chefe de investimentos altamente experiente.
    Você é responsável por revisar as análises fornecidas pelos analistas de ações e notícias, e fornecer uma recomendação final para o gerente de carteira com base em todas as informações disponíveis.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [18]:
recomendar_acao = Task(
    description="""
    Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.
    Se os relatórios não forem conclusivos, pode solicitar mais análises ou informações adicionais para chegar a uma recomendação mais informada.
    """,
    expected_output="Forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo.",
    agent=analista_chefe,
    context=[obter_carteira_cliente, obter_preco_acao, obter_noticias]
)

Agente 5 - Redator

In [19]:
redator = Agent(
    role="Redator de Relatórios de Investimentos",
    goal="Com base na recomendação final do analista chefe, redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, incluindo as análises de preço e notícias.",
    backstory="""
    Você é um redator experiente especializado em relatórios de investimentos.
    Sua tarefa é transformar a recomendação técnica do analista chefe em um relatório compreensível e útil para o cliente, destacando os pontos-chave e explicando as razões por trás da recomendação.
    Escreva em uma liguagem acessível, evitando jargões técnicos, para garantir que o cliente possa entender claramente a recomendação e os fatores que a influenciaram.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [20]:
escrever_boletim = Task(
    description="""
    Com base na recomendação final do analista chefe, redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, incluindo as análises de preço e notícias.
    Escreva em uma linguagem acessível, evitando jargões técnicos, para garantir que o cliente possa entender claramente a recomendação e os fatores que a influenciaram.
    Escreva com mais ou menos 6 paragráfos, destacando os pontos-chave da análise e explicando as razões por trás da recomendação de forma clara e objetiva.
    """,
    expected_output="""
    Redija um relatório claro e conciso para o cliente, explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, 
    incluindo as análises de preço e notícias. Escreva formatado como markdown, com títulos e subtítulos para organizar as informações de 
    forma clara e fácil de ler.
    Deve conter uma introdução explicando o objetivo do relatório, uma seção de análise de preço destacando as tendências e comparações com o preço pago,
    uma seção de análise de notícias explicando o impacto potencial das notícias no preço do ativo, e uma conclusão com a recomendação final de 
    compra, venda ou manutenção, resumindo os principais pontos que levaram a essa recomendação.
    """,
    agent=redator,
    context=[obter_preco_acao, obter_noticias, recomendar_acao]
)

In [21]:
crew = Crew(
    agents=[gerente_cliente, analista_acoes, analista_noticias, analista_chefe, redator],
    tasks=[obter_carteira_cliente, obter_preco_acao, obter_noticias, recomendar_acao, escrever_boletim],
    verbose=True,
    process = Process.hierarchical,
    full_output=True,
    share_crew=False,
    max_iter=5,
    manager_llm=llm
)

In [22]:
result = crew.kickoff(inputs={"ticket": "De a sua opinião sobre o ativo PETR4.SA, considerando as análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a carteira do cliente."})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3ea67651-7fd6-4ba2-98fc-d122973933af                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: ,                                                                                                        │
│      Use a pergunta do cliente e encontre o ativo De a sua opinião sobre o ativo PETR4.SA, considerando as      │
│  análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a        │
│  carteira do cliente. no arquivo CSV.                                                                           │
│      Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o       │
│  número total de ações em posse.                                                                                │
│                                                                                                                 │
│  ID: b84b213e-2388-49b0-b522-ad4596bc9e4a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: ,                                                                                                        │
│      Use a pergunta do cliente e encontre o ativo De a sua opinião sobre o ativo PETR4.SA, considerando as      │
│  análises de preço e notícias recentes, e forneça uma recomendação de compra, venda ou manutenção para a        │
│  carteira do cliente. no arquivo CSV.                                                                           │
│      Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o       │
│  número total de ações em posse.                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_csvs_content                                                                                    │
│  Args: {'search_query': 'PETR4.SA'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_csvs_content executed with result: Relevant Content:
Headers: Código | Nome do Ativo | Setor | Preço Atual (R$) | Preço Médio (R$) | Total de Ações
--------------------------------------------------
Row 
Row 1: Código: ABEV3 | Nome do ...


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: # Relatório de Investimento – Ações PETR4.SA                                                           │
│                                                                                                                 │
│  ## Introdução                                                                                                  │
│                                                                                                                 │
│  O objetivo deste relatório é apresentar uma análise clara e objetiva sobre a recomendação de investimento nas  │
│  ações PETR4.SA, considerando o desempenho recente, o cenário atual e as notícias relevantes que influenciam o  │
│  valor dessas ações. Com base na avaliação do analista chefe, explicaremos os motivos para a recomendação de    │
│  venda parcial e manutenção das ações restantes.                                                                │
│                                                                                                                 │
│  ## Análise de Preço                                                                                            │
│                                                                                                                 │
│  Você possui atualmente 150 ações da PETR4.SA adquiridas a um preço médio de R$ 27,50 por ação. O preço atual   │
│  dessas ações é de R$ 47,00, o que representa uma valorização expressiva de aproximadamente 70%. Essa alta      │
│  significativa indica que o investimento teve um bom desempenho até o momento.                                  │
│                                                                                                                 │
│  Além disso, a tendência das ações ainda é positiva, sustentada por um mercado de energia favorável. O setor    │
│  tem apresentado bom fluxo e a Petrobras conta com perspectivas econômicas robustas que podem continuar         │
│  impulsionando o preço no futuro.                                                                               │
│                                                                                                                 │
│  ## Análise de Notícias e Fatores Externos                                                                      │
│                                                                                                                 │
│  Diversos fatores recentes impactam positivamente as ações PETR4.SA:                                            │
│                                                                                                                 │
│  - **Investimentos Aumentados:** A Petrobras ampliou seus investimentos em exploração e infraestrutura,         │
│  fortalecendo sua capacidade produtiva.                                                                         │
│  - **Preços Elevados do Petróleo:** O mercado global registra preços do petróleo em níveis elevados, o que      │
│  favorece a receita da empresa.                                                                                 │
│  - **Dividendos Consistentes:** A companhia tem mantido o pagamento regular de dividendos, beneficiando os      │
│  acionistas com renda passiva.                                                                                  │
│  - **Avanços em Sustentabilidade:** Projetos e ações voltados à sustentabilidade reforçam a imagem da empresa   │
│  e seu compromisso com práticas responsáveis, o que pod

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Relatório de Investimento – Ações PETR4.SA                                                                   │
│                                                                                                                 │
│  ## Introdução                                                                                                  │
│                                                                                                                 │
│  O objetivo deste relatório é apresentar uma análise clara e objetiva sobre a recomendação de investimento nas  │
│  ações PETR4.SA, considerando o desempenho recente, o cenário atual e as notícias relevantes que influenciam o  │
│  valor dessas ações. Com base na avaliação do analista chefe, explicaremos os motivos para a recomendação de    │
│  venda parcial e manutenção das ações restantes.                                                                │
│                                                                                                                 │
│  ## Análise de Preço                                                                                            │
│                                                                                                                 │
│  Você possui atualmente 150 ações da PETR4.SA adquiridas a um preço médio de R$ 27,50 por ação. O preço atual   │
│  dessas ações é de R$ 47,00, o que representa uma valorização expressiva de aproximadamente 70%. Essa alta      │
│  significativa indica que o investimento teve um bom desempenho até o momento.                                  │
│                                                                                                                 │
│  Além disso, a tendência das ações ainda é positiva, sustentada por um mercado de energia favorável. O setor    │
│  tem apresentado bom fluxo e a Petrobras conta com perspectivas econômicas robustas que podem continuar         │
│  impulsionando o preço no futuro.                                                                               │
│                                                                                                                 │
│  ## Análise de Notícias e Fatores Externos                                                                      │
│                                                                                                                 │
│  Diversos fatores recentes impactam positivamente as ações PETR4.SA:                                            │
│                                                                                                                 │
│  - **Investimentos Aumentados:** A Petrobras ampliou seus investimentos em exploração e infraestrutura,         │
│  fortalecendo sua capacidade produtiva.                                                                         │
│  - **Preços Elevados do Petróleo:** O mercado global registra preços do petróleo em níveis elevados, o que      │
│  favorece a receita da empresa.                                                                                 │
│  - **Dividendos Consistentes:** A companhia tem mantido o pagamento regular de dividendos, beneficiando os      │
│  acionistas com renda passiva.                                                                                  │
│  - **Avanços em Sustentabilidade:** Projetos e ações vo

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Com base na recomendação final do analista chefe, redija um relatório claro e conciso para o cliente,      │
│  explicando a recomendação de compra, venda ou manutenção, e os motivos por trás dela, incluindo as análises    │
│  de preço e notícias.                                                                                           │
│      Escreva em uma linguagem acessível, evitando jargões técnicos, para garantir que o cliente possa entender  │
│  claramente a recomendação e os fatores que a influenciaram.                                                    │
│      Escreva com mais ou menos 6 paragráfos, destacando os pontos-chave da análise e explicando as razões por   │
│  trás da recomendação de forma clara e objetiva.                                                                │
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3ea67651-7fd6-4ba2-98fc-d122973933af                                                                       │
│  Final Output: # Relatório de Investimento – Ações PETR4.SA                                                     │
│                                                                                                                 │
│  ## Introdução                                                                                                  │
│                                                                                                                 │
│  O objetivo deste relatório é apresentar uma análise clara e objetiva sobre a recomendação de investimento nas  │
│  ações PETR4.SA, considerando o desempenho recente, o cenário atual e as notícias relevantes que influenciam o  │
│  valor dessas ações. Com base na avaliação do analista chefe, explicaremos os motivos para a recomendação de    │
│  venda parcial e manutenção das ações restantes.                                                                │
│                                                                                                                 │
│  ## Análise de Preço                                                                                            │
│                                                                                                                 │
│  Você possui atualmente 150 ações da PETR4.SA adquiridas a um preço médio de R$ 27,50 por ação. O preço atual   │
│  dessas ações é de R$ 47,00, o que representa uma valorização expressiva de aproximadamente 70%. Essa alta      │
│  significativa indica que o investimento teve um bom desempenho até o momento.                                  │
│                                                                                                                 │
│  Além disso, a tendência das ações ainda é positiva, sustentada por um mercado de energia favorável. O setor    │
│  tem apresentado bom fluxo e a Petrobras conta com perspectivas econômicas robustas que podem continuar         │
│  impulsionando o preço no futuro.                                                                               │
│                                                                                                                 │
│  ## Análise de Notícias e Fatores Externos                                                                      │
│                                                                                                                 │
│  Diversos fatores recentes impactam positivamente as ações PETR4.SA:                                            │
│                                                                                                                 │
│  - **Investimentos Aumentados:** A Petrobras ampliou seus investimentos em exploração e infraestrutura,         │
│  fortalecendo sua capacidade produtiva.                                                                         │
│  - **Preços Elevados do Petróleo:** O mercado global registra preços do petróleo em níveis elevados, o que      │
│  favorece a receita da empresa.                                                                                 │
│  - **Dividendos Consistentes:** A companhia tem mantido o pagamento regular de dividendos, beneficiando os      │
│  acionistas com renda passiva.                                                                                  │
│  - **Avanços em Sustentabilidade:** Projetos e ações v

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
print(result.raw)

# Relatório de Investimento – Ações PETR4.SA

## Introdução

O objetivo deste relatório é apresentar uma análise clara e objetiva sobre a recomendação de investimento nas ações PETR4.SA, considerando o desempenho recente, o cenário atual e as notícias relevantes que influenciam o valor dessas ações. Com base na avaliação do analista chefe, explicaremos os motivos para a recomendação de venda parcial e manutenção das ações restantes.

## Análise de Preço

Você possui atualmente 150 ações da PETR4.SA adquiridas a um preço médio de R$ 27,50 por ação. O preço atual dessas ações é de R$ 47,00, o que representa uma valorização expressiva de aproximadamente 70%. Essa alta significativa indica que o investimento teve um bom desempenho até o momento.

Além disso, a tendência das ações ainda é positiva, sustentada por um mercado de energia favorável. O setor tem apresentado bom fluxo e a Petrobras conta com perspectivas econômicas robustas que podem continuar impulsionando o preço no futuro.

##